# 03 | Entitlements, baskets and dynamic pricing

**Author: Chanakya**

Compare what each customer can actually consume. Prices are date- and channel-specific observations or labelled case inputs. Intended viewing baskets are scenarios, not observed customer demand.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='03_offers_and_baskets'
shared.ACTIVE_SOURCES=['fancode_monthly', 'fancode_monthly_final_v5', 'fancode_yearly', 'fancode_yearly_rendered', 'jiohotstar_2026_price_release', 'sonyliv_subscription', 'v3_tennistv_india_iap']

## 1. Audited product menu
No monthly ATP entitlement is assumed. The yearly description explicitly includes tennis. Competitors differ in rights, devices and term. Comparing rupees per month does not make their content interchangeable.

In [ ]:
offers=[
('FanCode tournament','ATP',79,99,'tournament','case input','supplied_BGCC_FANCODE_R3','PDF p11. Current checkout unverified'),
('FanCode season','ATP',399,399,'2026 season','case input','supplied_BGCC_FANCODE_R3','Declining late-season price not observed'),
('FanCode monthly list','ATP uncertain',199,199,'30 days','public observation','fancode_monthly_final_v5','Explicit F1/MotoGP, ATP unresolved'),
('FanCode monthly promo','ATP uncertain',116,116,'30 days','public observation','fancode_monthly_final_v5','SEPT10 applied, renewal price not verified'),
('FanCode yearly list','Portfolio including tennis',999,999,'365 days','public observation','fancode_yearly_rendered','Device and scope per captured description'),
('FanCode yearly promo','Portfolio including tennis',899,899,'365 days','public observation','fancode_yearly_rendered','Coupon snapshot, not permanent policy'),
('Sony LIV Premium monthly','Different rights basket',399,399,'month','public listing','sonyliv_subscription','Not an ATP-equivalent substitute'),
('Sony LIV Premium annual','Different rights basket',1499,1499,'year','public listing','sonyliv_subscription','Not an ATP-equivalent substitute'),
('Sony LIV mobile annual','Mobile only',699,699,'year','public listing','sonyliv_subscription','Device restriction matters'),
('Tennis TV monthly IAP','ATP product',449,449,'month','India App Store listing','v3_tennistv_india_iap','Not web checkout or purchased entitlement'),
('Tennis TV six-month IAP','ATP product',2500,2500,'six months','India App Store listing','v3_tennistv_india_iap','Not directly equivalent to FanCode season'),
('Tennis TV annual IAP','ATP product',4499,4499,'year','India App Store listing','v3_tennistv_india_iap','Grand Slams excluded from ATP comparison'),
('JioHotstar Mobile','Different rights basket',79,79,'month','January announcement','jiohotstar_2026_price_release','Dated launch price, not September checkout'),
('JioHotstar Super','Different rights basket',149,149,'month','January announcement','jiohotstar_2026_price_release','Rights basket differs'),
('JioHotstar Premium','Different rights basket',299,299,'month','January announcement','jiohotstar_2026_price_release','Rights basket differs')]
offers=pd.DataFrame(offers,columns=['product','scope','low_price','high_price','validity','evidence_type','source_id','limitation']);table(offers,'offers',True);display(table(offers,'03_offer_audit'))
print('Two INR99 tournament passes cost INR198. Monthly list costs INR1 more, while observed promotion costs INR82 less, only if ATP is included and dates fit the term.')

## 2. Minimum-cost coverage over explicit dated baskets
Solve a small weighted set-cover problem exactly using a bitmask dynamic programme. Each required viewing date must be covered. Monthly validity is 30 days from purchase, yearly 365. The buyer can start a pass on a demand date. Tournament access covers that event. Existing ownership is modelled as zero-cost coverage. This is product arithmetic, not an elasticity or choice model.

In [ ]:
from itertools import product
future=read('atp_future_windows');f1=read('f1_races')
# Dated ATP occasions are event-window endpoints, not promised match times.
selected=[]
for name in ['Chengdu','Tokyo','Shanghai','Basel','Paris','Nitto']:
 r=future[future.event.str.contains(name)].iloc[0];selected.append((name,pd.Timestamp(r.end_date),'ATP'))
races=[(r.event,pd.Timestamp(r.start_ist).tz_localize(None).normalize(),'F1') for r in f1.itertuples() if pd.Timestamp(r.start_ist)>=pd.Timestamp('2026-09-23',tz='Asia/Kolkata')]
baskets={'One tennis event':selected[:1],'Two tennis events within 30d':[selected[1],selected[2]],'Late-season tennis six':selected,'Tennis plus two race weekends':selected[:3]+races[:2],'Portfolio to season end':selected+races}
def cheapest(demands,monthly_atp,monthly_price,yearly_price,owned=False,tournament_price=99):
 n=len(demands);full=(1<<n)-1
 if owned:return 0.,'Existing covering entitlement'
 candidates=[]
 for j,(event,date,sport) in enumerate(demands):
  if sport=='ATP':candidates.append((tournament_price,1<<j,'Tournament '+event))
 atp_mask=sum(1<<j for j,d in enumerate(demands) if d[2]=='ATP')
 candidates.append((399,atp_mask,'ATP season'))
 for _,start,_ in demands:
  mask=sum(1<<j for j,(_,dt,sp) in enumerate(demands) if start<=dt<start+pd.Timedelta(days=30) and (sp!='ATP' or monthly_atp))
  candidates.append((monthly_price,mask,'Monthly '+start.date().isoformat()))
  ymask=sum(1<<j for j,(_,dt,_) in enumerate(demands) if start<=dt<start+pd.Timedelta(days=365))
  candidates.append((yearly_price,ymask,'Yearly '+start.date().isoformat()))
 dp={0:(0.,[])}
 for mask in range(full+1):
  if mask not in dp:continue
  for price,cover,label in candidates:
   new=mask|cover
   if new!=mask and (new not in dp or dp[mask][0]+price<dp[new][0]):dp[new]=(dp[mask][0]+price,dp[mask][1]+[label])
 return dp[full][0],'; '.join(dp[full][1])
rows=[];basketrows=[]
for name,demands in baskets.items():
 for event,date,sport in demands:basketrows.append(dict(basket=name,event=event,date=date.date(),sport=sport,interpretation='Scenario intention, source-dated event occasion'))
 for inclusion,mp,yp,tp in product([False,True],[116,199],[899,999],[79,89,99]):
  cost,choice=cheapest(demands,inclusion,mp,yp,tournament_price=tp);rows.append(dict(basket=name,monthly_atp=inclusion,monthly_price=mp,yearly_price=yp,tournament_price=tp,total_cost=cost,choice=choice))
result=pd.DataFrame(rows);table(pd.DataFrame(basketrows),'03_basket_definitions');table(result,'03_minimum_cost_baskets');table(result,'basket_results',True)
base=result[(result.yearly_price==999)&(result.tournament_price==99)];display(base[['basket','monthly_atp','monthly_price','total_cost','choice']])
plot=base.assign(scenario=base.monthly_atp.map({False:'ATP excluded',True:'ATP included'})+' / INR'+base.monthly_price.astype(str)).pivot(index='basket',columns='scenario',values='total_cost')
plot.plot.barh(figsize=(11,5));plt.xlabel('Minimum new payment (INR)');plt.title('The same viewing basket can imply a different pass');plt.legend(fontsize=8);fig('03_basket_switch_map','Scenario intentions on source-dated events. Monthly ATP inclusion unresolved. No customer demand inferred.')

## 3. Transparent pricing rules and dominance checks
Test tier, timing and remaining season sequentially. A player pass cannot promise unplayed matches. Replay INR39 is a feasibility-dependent test, not an existing SKU. Remaining season is priced here using event-start opportunities as an explicitly coarse inventory proxy. It is not viewer value or a precise remaining-live-days measure.

In [ ]:
events=read('atp_event_starts');regular=events[events.tier.isin(['ATP 250','ATP 500','ATP MASTERS 1000']) & events.start_date.notna()].copy();regular['start_date']=pd.to_datetime(regular.start_date)
curve=[]
for date in pd.date_range('2026-01-01','2026-11-23',freq='7D'):
 remaining=int((regular.start_date>=date).sum());inventory_price=399*remaining/len(regular);curve.append(dict(date=date,remaining_event_starts=remaining,reference_price=399,raw_inventory_price=inventory_price,candidate_rounded_price=round(min(inventory_price,99*remaining)),proxy='Equal-weight event starts, excludes ongoing event replays'))
curve=pd.DataFrame(curve);table(curve,'03_remaining_inventory_price')
plt.figure(figsize=(10,3.8));plt.step(curve.date,curve.candidate_rounded_price,where='post');plt.ylabel('Candidate season price (INR)');plt.title('An inventory-based declining price is a test rule, not WTP');fig('03_remaining_season_rule','Guide dates, equal event-start weights. Current declining season checkout is unknown. Validate before testing.')
formats=pd.DataFrame([
('Single match','Proposed INR49 test','One identified match, replay expiry specified before launch','Do not lead with paid acquisition at INR150–200 CAC. Test total cohort contribution and displacement.'),
('Tournament','Case INR79–99','Covered event, scope and replays must be clear','Default low-commitment entry when it is the cheapest covering basket.'),
('Season','Case INR399, declines late season','Remaining ATP coverage, not 365 days','Use actual current price and intended remaining events.'),
('Replay rental','Proposed INR39 test','One selected replay, candidate 48-hour viewing window after activation','Rights and playback enforcement unverified. Reject if substitution loss exceeds induced contribution.'),
('Player or event bundle','Price via existing minimum-cost basket first','Confirmed appearances or flexible event access, explicit withdrawal/refund treatment','Do not promise progression or invent a player premium from video views.'),
('Portfolio bundle','Observed monthly/yearly menu','Dates, devices and ATP inclusion checked','Monthly ATP scope is a two-branch analysis. Annual requires sufficient intended portfolio value.')],columns=['format','price_status','coverage_rule','lead_decision']);display(table(formats,'03_format_decisions'))
check('03_offers',{'two_events_list_comparison':2*99==198,'promotion_gap':198-116==82,'yearly_cannot_win_pure_six_tennis_at_list':cheapest(selected,False,199,999)[0]<=399,'ownership_suppresses_sale':cheapest(selected,True,199,999,owned=True)[0]==0,'adding_atp_access_never_raises_cost':bool((result.pivot(index=['basket','monthly_price','yearly_price','tournament_price'],columns='monthly_atp',values='total_cost')[True]<=result.pivot(index=['basket','monthly_price','yearly_price','tournament_price'],columns='monthly_atp',values='total_cost')[False]).all())})
report('03_offer_findings','Do not automatically push every multi-sport user to a yearly pass. The exact covering basket and existing entitlement decide. Short monthly access may dominate a tennis mini-bundle if ATP is included. Publish complete coverage and suppress redundant sales. Player-specific products require participation protection or refund terms. Test replay and credit on total portfolio contribution.')

## Source references
These IDs resolve to the preserved bodies, URLs and capture timestamps. Derived tables also retain row-level source IDs where applicable. Case inputs refer to the supplied brief, physical PDF pages 9–14. Review source files resolve through the review collection log. Scenario parameters are in analysis_config.

In [ ]:
references=source_table(['fancode_monthly', 'fancode_monthly_final_v5', 'fancode_yearly', 'fancode_yearly_rendered', 'jiohotstar_2026_price_release', 'sonyliv_subscription', 'v3_tennistv_india_iap'])
display(table(references,'03_source_references'))